# Basketball Position Classification

??? 3??, ????, ?? ????? ???(`SG`, `C`)? ?????. ???? ?? ??? ?? ??? ???? feature ??? ?? ??? ?? ???? ? ??? ?????.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

sns.set_theme(style="whitegrid")

train = pd.read_csv("data/basketball_train.csv")
test = pd.read_csv("data/basketball_test.csv")

train = train.drop(columns=[col for col in train.columns if col.startswith("Unnamed")])
test = test.drop(columns=[col for col in test.columns if col.startswith("Unnamed")])

print(train.shape, test.shape)
display(train.head())

## 1. ??? ??

????? `3P`, `TRB`, `BLK`? ??? ?????. SG? 3??, C? ????? ???? ??? ? ???? ???.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, ["3P", "TRB", "BLK"]):
    sns.boxplot(data=train, x="Pos", y=col, ax=ax)
    ax.set_title(f"{col} by position")
plt.tight_layout()

In [ ]:
sns.pairplot(train, vars=["3P", "TRB", "BLK"], hue="Pos", height=2.5)

## 2. ???

KNN? ?? ?? ????? StandardScaler? ?????. SVM? ?? ??? RBF ??? ?????.

In [ ]:
features = ["3P", "TRB", "BLK"]
X_train = train[features]
y_train = train["Pos"]
X_test = test[features]
y_test = test["Pos"]

models = {
    "KNN(k=3)": Pipeline([("scaler", StandardScaler()), ("model", KNeighborsClassifier(n_neighbors=3))]),
    "KNN(k=5)": Pipeline([("scaler", StandardScaler()), ("model", KNeighborsClassifier(n_neighbors=5))]),
    "SVM(linear)": Pipeline([("scaler", StandardScaler()), ("model", SVC(kernel="linear", random_state=42))]),
    "SVM(rbf)": Pipeline([("scaler", StandardScaler()), ("model", SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42))]),
}

rows = []
fitted = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    rows.append({"model": name, "test_accuracy": accuracy_score(y_test, pred)})
    fitted[name] = model

results = pd.DataFrame(rows).sort_values("test_accuracy", ascending=False)
display(results)

## 3. ?? ??

?? ?? accuracy? ?? ??? classification report? confusion matrix? ?????.

In [ ]:
best_name = results.iloc[0]["model"]
best_model = fitted[best_name]
pred = best_model.predict(X_test)

print(f"Best model: {best_name}")
print(classification_report(y_test, pred, zero_division=0))

ConfusionMatrixDisplay.from_predictions(y_test, pred, cmap="Greens")
plt.title(f"Confusion Matrix - {best_name}")
plt.show()

## 4. ??

?? ??????? ??? ??? ????? ??? ???? ?????. ? ??????? feature ??? ?? ??, KNN? SVM? ?? ???? ???? ???? ???? ? ??? ????.